# 30 · Revision-5 training runs — session A (T4 GPU; run sessions A and B at the same time)
200 runs (seeds 3–4 of the eight bottleneck models, both schedules; seeds 1–4 of full and unet, schedule 'train'), about 13 min each.
Two copies of this notebook (A and B) share the work through lock files. Every run is trained in its own Python process
(`p2_run_rev5.py`, same code path as notebooks 13 and 27), so GPU memory is released after each run (no out-of-memory build-up).
Each run is trained on the local disk of the runtime and copied to Drive once when it ends, so Drive Trash does not fill during training.
**Resumable:** after a disconnect simply **Run all** again; finished runs are skipped, an interrupted run restarts from epoch 0.
The prespecified text (cell 1) is stored with a UTC time stamp before the first run.

In [ ]:
# 1 · Settings, Drive, protocol record
SESSION = 'A'
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, json, time, shutil, subprocess
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CODE     = f'{PROJECT}/code'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
RES      = f'{PROJECT}/results'
OUT      = f'{RES}/p2_rev5'
ROOTS    = {('train', False): f'{RES}/p2_jit_runs_train', ('val', False): f'{RES}/p2_jit_runs_val',
            ('train', True): f'{RES}/p2_rev4_runs_train', ('val', True): f'{RES}/p2_rev4_runs_val'}
for d in [OUT] + list(ROOTS.values()): os.makedirs(d, exist_ok=True)
PROTO = """Paper 2 - revision 5 (extension to five seeds and two further external test sets), written before any revision-5 run was trained
and before any model was evaluated on CSAW-S or CDD-CESM.

A. New training runs. Code, data (400 CBIS-DDSM mass crops, folds.csv), settings and evaluation exactly as for seeds 0-2: batch 16,
   Adam 1e-4, up to 200 epochs, best checkpoint and early stopping (patience 10, from epoch 15) on the validation loss, learning rate
   halved after four epochs without improvement of the training loss (schedule 'train') or of the validation loss (schedule 'val').
   (1) Seeds 3 and 4 of the eight bottleneck models (g_q, g_cl, g_id, g_tr, p_q, p_cl, p_id, p_tr), folds 0-4, both schedules
       (160 runs).
   (2) Seeds 1-4 of the two reference models (full, unet), folds 0-4, schedule 'train' (40 runs).
   Hardware: Google Colab, NVIDIA T4 only (the GPU type of seeds 0-2); a session that is not on a T4 does not start any run. Each run
   is trained in its own Python process. Python, TensorFlow, CUDA and GPU of every run are stored with the run (env.json); the
   TensorFlow version of the Colab image may differ from the one of seeds 0-2 and is reported. Runs keep their usual folders
   (g_tr and p_tr: p2_rev4_runs_<schedule>; all other models: p2_jit_runs_<schedule>). A run that stops because of an
   infrastructure error (disconnect, out of memory) is restarted from epoch 0; a run whose validation loss is never finite is
   recorded as failed and reported. All 200 runs are reported; none is discarded or repeated for its result.

B. Test sets (no fine-tuning; threshold 0.5; per image Dice, IoU, HD95 and an empty-prediction flag). Every saved model of seeds
   0-4 is evaluated on every set with the same code (notebook 31); per crop and seed, Dice is averaged over the five fold models.
   (1) CBIS-DDSM test folds (out-of-fold, as in the study).
   (2) Frozen CBIS-DDSM cohort (400 mass ROIs of patients outside the study, revision 4).
   (3) DMID (revision-4 crops).
   (4) CSAW-S test set (Zenodo 10.5281/zenodo.4030660): reference = pixels outlined as cancer by at least two of the three
       radiologists; one lesion = one 8-connected component of the consensus mask with area >= 0.5% of the image; crops with the
       study's randomized protocol (scale 1.5-3.0 x lesion box, shift 0.8, seed from the MD5 of the lesion id, 256 x 256) and the
       study's 1st-99.5th percentile intensity window (notebook 29). Per-radiologist masks are cropped with the same window.
   (5) CDD-CESM (TCIA, CC BY 4.0): low-energy (DM) images only; one lesion = one hand-drawn region whose finding type in the data
       set's own annotation table is a mass (benign and malignant); if the table gives finding types per image rather than per
       region, the regions of images whose findings include a mass and no other region-forming finding (calcification,
       architectural distortion, asymmetry) are used; all other regions are excluded; region shapes are rasterized as drawn
       (polygon, ellipse, circle, rectangle); crops and window as in (4). The lesion inventory (images, lesions, patients,
       exclusions by reason) is printed and stored before any model is evaluated.
   For seeds 0-2, re-evaluation on (2) and (3) must reproduce the stored revision-4 values (largest absolute per-image Dice
   difference reported); otherwise the stored values are kept and the difference is reported.

C. Estimands and analysis (seeds 0-4; per-image Dice).
   Primary: the nine model contrasts of Tables VI-VII under each schedule and the circuit - trigonometric contrast at each placement
   under each schedule, on the CBIS-DDSM test folds. Crossed hierarchical bootstrap over patients and seeds (10,000 replicates,
   rng(0)); Holm within the nine contrasts of each schedule and within the four circuit - trigonometric contrasts.
   Equivalence: a contrast is called equivalent when its 95% interval lies within -0.02 to +0.02 Dice; a contrast whose interval
   excludes 0 is called a difference; otherwise inconclusive.
   Secondary: the same contrasts on (2)-(5) (Holm within each set and schedule; CSAW-S consensus reference primary, per-radiologist
   references as sensitivity); model levels; full and unet as reference models; empty-prediction rates and HD95; code-replacement
   drops of the circuit and trigonometric models.
   Seed sensitivity: seeds 0-2 versus 3-4 (difference of the contrast between the two seed blocks with its interval), leave one
   seed out, and the contrasts with seeds 0-2 only (the previously reported values) shown next to the five-seed values.
   The five-seed results are the main results of this revision; the three-seed results remain reported.
   No further seeds, models, schedules or test sets are added after this text is stored.
All runs and all results are reported; none is discarded."""
proto = f'{OUT}/PROTOCOL_rev5_prespecified.txt'
if not os.path.exists(proto):
    open(proto, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
assert open(proto).read().startswith(PROTO), 'protocol file on Drive differs from this notebook'
print(open(proto).read().splitlines()[-1])

In [ ]:
# 2 · GPU check (T4 only), runner script, local copy of the CBIS-DDSM crops
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
assert 'T4' in gpu, f'not a T4 ({gpu}): Runtime -> Change runtime type -> T4 GPU, then Run all'
for f in ('qseg2.py', 'qseg2_rev4.py'):
    assert os.path.exists(f'{CODE}/{f}'), f'{CODE}/{f} missing'
# qseg package: located exactly as in notebooks 13 and 27 (project code folder first, otherwise the first qseg/quantum_sim.py on Drive)
import glob, hashlib
QSEG_DIR = CODE
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
QSEG_MD5 = {os.path.basename(f): hashlib.md5(open(f, 'rb').read()).hexdigest() for f in sorted(glob.glob(f'{QSEG_DIR}/qseg/*.py'))}
json.dump(dict(qseg_dir=QSEG_DIR, md5=QSEG_MD5), open(f'{OUT}/qseg_used_{SESSION}.json', 'w'), indent=1)
print('qseg package:', QSEG_DIR, '| files:', len(QSEG_MD5))
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([QSEG_DIR, CODE] + ([os.environ['PYTHONPATH']] if os.environ.get('PYTHONPATH') else [])))
RUNNER = f'{CODE}/p2_run_rev5.py'
open(RUNNER, 'w').write('"""Paper 2: one training run from the command line (same code path as notebooks 13 and 27; for batch/SLURM use).\n\n    python p2_run.py --model g_q --fold 0 --seed 3 --schedule train \\\n        --cbis /path/data/processed/cbis_mass_256_jit --out /path/results\n\nRuns go to <out>/p2_jit_runs_<schedule>/<model>_fold<k>_seed<s>/ (or to --root) exactly as in the notebooks (summary.json, history.csv,\nper-image CSV, best.weights.h5 without optimizer state). A finished run (summary.json present) is skipped, so a job array can\nsimply be resubmitted. Settings: Config defaults of the study with max_epochs=200; learning rate halved after 4 epochs without\nimprovement of the training loss (schedule \'train\') or of the validation loss (schedule \'val\'). The software and GPU of every\nrun are written to <run>/env.json.\n"""\nimport argparse, json, os, platform, subprocess, sys, time\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\n\nimport numpy as np\nimport tensorflow as tf\nimport qseg2\nimport qseg2_rev4\nfrom qseg.config import Config\n\nqseg2_rev4.patch()                         # adds g_tr / p_tr\n\n\ndef use_schedule(schedule):\n    """As notebook 27, cell 1: ReduceLROnPlateau watches \'loss\' (train) or \'val_loss\' (val)."""\n    base = tf.keras.callbacks.ReduceLROnPlateau\n    while getattr(base, \'_p2_switch\', False):\n        base = base.__mro__[1]\n    monitor = {\'train\': \'loss\', \'val\': \'val_loss\'}[schedule]\n\n    class SwitchPlateau(base):\n        _p2_switch = True\n\n        def __init__(self, monitor_=None, **kw):\n            kw.pop(\'monitor\', None)\n            super().__init__(monitor=monitor, **kw)\n\n    tf.keras.callbacks.ReduceLROnPlateau = SwitchPlateau\n\n\ndef shrink_weights(cfg, run_dir, name):\n    """As notebook 27, cell 2: weight file without optimizer state, verified weight by weight."""\n    big, small = f\'{run_dir}/best.weights.h5\', f\'{run_dir}/best_small.weights.h5\'\n    if not os.path.exists(big) or os.path.exists(f\'{run_dir}/.shrunk\'):\n        return\n    if os.path.getsize(big) < 60e6:\n        open(f\'{run_dir}/.shrunk\', \'w\').close(); return\n    shape = (cfg.image_size, cfg.image_size, cfg.in_channels)\n    tf.keras.backend.clear_session()\n    m1 = qseg2.build_p2_model(name, shape, cfg.n_qubits, cfg.n_q_layers); m1.load_weights(big); m1.save_weights(small)\n    m2 = qseg2.build_p2_model(name, shape, cfg.n_qubits, cfg.n_q_layers); m2.load_weights(small)\n    assert all(np.array_equal(a, b) for a, b in zip(m1.get_weights(), m2.get_weights())), f\'{run_dir}: weight check failed\'\n    os.replace(small, big); open(f\'{run_dir}/.shrunk\', \'w\').close()\n\n\ndef env_info():\n    try:\n        gpu = subprocess.run([\'nvidia-smi\', \'--query-gpu=name,driver_version\', \'--format=csv,noheader\'],\n                             capture_output=True, text=True, timeout=30).stdout.strip()\n    except Exception:\n        gpu = \'unknown\'\n    b = tf.sysconfig.get_build_info()\n    return dict(python=platform.python_version(), tensorflow=tf.__version__, keras=tf.keras.__version__ if hasattr(tf.keras, \'__version__\') else \'\',\n                cuda=b.get(\'cuda_version\'), cudnn=b.get(\'cudnn_version\'), gpu=gpu, host=platform.node())\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--model\', required=True, choices=sorted(qseg2.MODELS2))\n    ap.add_argument(\'--fold\', type=int, required=True, choices=range(5))\n    ap.add_argument(\'--seed\', type=int, required=True)\n    ap.add_argument(\'--schedule\', required=True, choices=(\'train\', \'val\'))\n    ap.add_argument(\'--cbis\', required=True, help=\'folder with folds.csv, images/, masks/\')\n    ap.add_argument(\'--out\', required=True, help=\'results root\')\n    ap.add_argument(\'--root\', default=None, help=\'run folder root (default <out>/p2_jit_runs_<schedule>)\')\n    a = ap.parse_args()\n\n    root = a.root or os.path.join(a.out, f\'p2_jit_runs_{a.schedule}\')\n    run = f\'{a.model}_fold{a.fold}_seed{a.seed}\'\n    d = os.path.join(root, run)\n    if os.path.exists(os.path.join(d, \'summary.json\')):\n        print(\'already finished:\', run); return\n    os.makedirs(d, exist_ok=True)\n    json.dump(env_info(), open(os.path.join(d, \'env.json\'), \'w\'), indent=2)\n\n    use_schedule(a.schedule)\n    cfg = Config(work_dir=a.out, max_epochs=200)\n    if not os.path.exists(os.path.join(root, \'train_config.json\')):    # never overwrite the record of earlier runs\n        cfg.to_json(os.path.join(root, \'train_config.json\'))\n    if not os.path.exists(os.path.join(root, \'schedule.json\')):\n        json.dump({\'schedule\': a.schedule}, open(os.path.join(root, \'schedule.json\'), \'w\'))\n    data = qseg2.load_cbis(a.cbis)\n    t0 = time.time()\n    s = qseg2.run_fold(cfg, data, a.model, a.fold, a.seed, out_root=root, min_epochs=15, verbose=2)\n    shrink_weights(cfg, d, a.model)\n    print(f\'{run} [{a.schedule}]: Dice {s["dice_per_image"]:.4f}, {s["epochs_run"]} epochs, {(time.time() - t0) / 60:.1f} min\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
LOCAL = '/content/cbis_mass_256_jit'
if not os.path.exists(f'{LOCAL}/folds.csv'): shutil.copytree(CBIS_DIR, LOCAL)
print('crops:', len(os.listdir(f'{LOCAL}/images')), '| runner written')

In [ ]:
# 3 · Training (sessions share the work through lock files; one process per run)
MODELS8 = ('g_q', 'g_cl', 'g_id', 'g_tr', 'p_q', 'p_cl', 'p_id', 'p_tr')
RUNS = [(m, f, s, 'train') for s in (1, 2) for f in range(5) for m in ('full', 'unet')]
RUNS += [(m, f, s, sch) for s in (3, 4) for sch in ('train', 'val') for f in range(5) for m in MODELS8]
RUNS += [(m, f, s, 'train') for s in (3, 4) for f in range(5) for m in ('full', 'unet')]
assert len(RUNS) == 200 and len(set(RUNS)) == 200
def root_of(m, sch): return ROOTS[(sch, m in ('g_tr', 'p_tr'))]
def rdir(m, f, s, sch): return f'{root_of(m, sch)}/{m}_fold{f}_seed{s}'
def finished(r): return os.path.exists(rdir(*r) + '/summary.json') or os.path.exists(rdir(*r) + '/failed.json')
LOCK_H = 2
def claim(r):
    d = rdir(*r)
    if finished(r): return False
    lk = f'{d}/.lock'
    if os.path.exists(lk):
        try: who, t = open(lk).read().split(); t = float(t)
        except Exception: who, t = SESSION, 0.0
        if who != SESSION and time.time() - t < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}'); time.sleep(30)
    try: return open(lk).read().split()[0] == SESSION
    except Exception: return False
WORK = '/content/p2_work'     # each run is trained on the local disk and copied to Drive once at the end:
                              # no checkpoint or history file is overwritten on Drive during training (keeps Drive Trash small)
def train(r):
    m, f, s, sch = r; log = f'/content/log_{m}_{f}_{s}_{sch}.txt'
    lroot = f'{WORK}/{os.path.basename(root_of(m, sch))}'; ldir = f'{lroot}/{m}_fold{f}_seed{s}'
    shutil.rmtree(ldir, ignore_errors=True); os.makedirs(lroot, exist_ok=True)
    cmd = [sys.executable, RUNNER, '--model', m, '--fold', str(f), '--seed', str(s), '--schedule', sch,
           '--cbis', LOCAL, '--out', RES, '--root', lroot]
    with open(log, 'w') as fh: rc = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=ENV).returncode
    d = rdir(*r); os.makedirs(d, exist_ok=True)
    if os.path.isdir(ldir):
        names = sorted(os.listdir(ldir), key=lambda n: n == 'summary.json')      # summary.json last = run complete on Drive
        for n in names:
            src = f'{ldir}/{n}'
            if os.path.isdir(src): shutil.copytree(src, f'{d}/{n}', dirs_exist_ok=True)
            else: shutil.copy2(src, f'{d}/{n}')
        shutil.rmtree(ldir, ignore_errors=True)
    shutil.copy(log, d + f'/train_log_attempt{int(time.time())}.txt')
    return rc, open(log).read()[-3000:]
order = RUNS if SESSION == 'A' else RUNS[::-1]
for r in order:
    if not claim(r): continue
    m, f, s, sch = r; t0 = time.time()
    rc, tail = train(r)
    if rc != 0 and not os.path.exists(rdir(*r) + '/summary.json'):
        if 'validation loss never finite' in tail:
            json.dump(dict(model=m, fold=f, seed=s, schedule=sch, failed='validation loss never finite'), open(rdir(*r) + '/failed.json', 'w'))
        else:
            print(f'   {m} f{f} s{s} [{sch}]: process error (code {rc}), retrying once from epoch 0 ...\n' + tail[-1500:], flush=True)
            rc, tail = train(r)
            if rc != 0 and not os.path.exists(rdir(*r) + '/summary.json'):
                print('   second failure - stopping this session so that the error can be checked:\n' + tail, flush=True)
                raise SystemExit(1)
    done = sum(os.path.exists(rdir(*q) + '/summary.json') for q in RUNS)
    if os.path.exists(rdir(*r) + '/summary.json'):
        S = json.load(open(rdir(*r) + '/summary.json'))
        print(time.strftime('%H:%M UTC', time.gmtime()), f"| {m} fold {f} seed {s} [{sch}]: Dice {S['dice_per_image']:.4f}, "
              f"{S['epochs_run']} ep (best {S['best_epoch']}), {(time.time() - t0) / 60:.1f} min | finished {done} / {len(RUNS)}", flush=True)
    else:
        print(time.strftime('%H:%M UTC', time.gmtime()), f'| {m} fold {f} seed {s} [{sch}]: FAILED (recorded) | finished {done} / {len(RUNS)}', flush=True)
left = [r for r in RUNS if not finished(r)]
print(f'session {SESSION} has nothing left to start; {len(left)} run(s) still running in the other session')

In [ ]:
# 4 · Overview of the revision-5 runs
import pandas as pd
rows = [dict(json.load(open(rdir(*r) + '/summary.json')), schedule=r[3]) for r in RUNS if os.path.exists(rdir(*r) + '/summary.json')]
ov = pd.DataFrame(rows); ov.to_csv(f'{OUT}/overview_rev5_{SESSION}.csv', index=False)
print(len(ov), 'of', len(RUNS), 'finished;', sum(os.path.exists(rdir(*r) + '/failed.json') for r in RUNS), 'failed')
if len(ov): print(ov.groupby(['schedule', 'model'])[['dice_per_image', 'epochs_run', 'best_epoch']].mean().round(4).to_string())